# Performance benchmarks for joining on nested key columns

In [2]:
import duckdb
import numpy as np
import polars as pl
from tests.conftest import TestData

In [3]:
con = duckdb.connect()

### Question

What is the performance impact of using nested structs instead of flat columns for joins? This matters for our file format, in particular,
how we store columns that contain primary and secondary keys.

### Lessons learned
- structs are not free, see https://dl.acm.org/doi/10.1145/3725329
- the benchmarks are special cases because all the set of identifiers is the same in both tables of the join. We might want to generalize at some point. 


## Small in-memory data

In [4]:
data1 = {"id_1": [1, 2], "id_2": [10, 11], "a": [1, 2], "b": [3, 4]}
data2 = {"id_1": [2, 1], "id_2": [11, 10], "x": ["hello", "bye"], "y": ["how", "are you"]}
data1 = pl.DataFrame(data1)
data2 = pl.DataFrame(data2)

In [5]:
def data_with_struct_id(data1, data2):
    data1_struct = data1.with_columns(pl.struct(pl.col(["id_1", "id_2"])).alias("id")).drop(pl.col(["id_1", "id_2"]))
    data2_struct = data2.with_columns(pl.struct(pl.col(["id_1", "id_2"])).alias("id")).drop(pl.col(["id_1", "id_2"]))
    return data1_struct, data2_struct

In [6]:
def time_duck(con, data1, data2, condition):
    con.execute("DROP TABLE IF EXISTS t1")
    con.execute("DROP TABLE IF EXISTS t2")
    from_file = isinstance(data1, str)
    if from_file:
        sql = f"""
            SELECT * FROM read_parquet('{data1}') AS t1
            INNER JOIN read_parquet('{data2}') AS t2
            USING ({condition})
        """
        t_duck = %timeit -o con.execute(sql)
    else:
        con.execute("CREATE TABLE t1 AS SELECT * FROM data1")
        con.execute("CREATE TABLE t2 AS SELECT * FROM data2")
        t_duck = %timeit -o con.execute(f"SELECT * FROM t1 INNER JOIN t2 using ({condition})")

    return t_duck

In [7]:
def compare_timing(data1: pl.DataFrame | str, data2: pl.DataFrame | str, con):
    """Compare timing across duck/polars and struct/non-struct."""
    result = {}
    from_file = False
    if isinstance(data1, str):
        from_file = True
        file1 = data1
        file2 = data2
        data1 = pl.scan_parquet(data1)
        data2 = pl.scan_parquet(data2)

    if not from_file:
        t_polars = %timeit -o data1.join(data2, on = ["id_1", "id_2"])
        t_duck = time_duck(con, data1, data2, "id_1, id_2")
    else:
        t_polars = %timeit -o data1.join(data2, on = ["id_1", "id_2"]).collect()
        t_duck = time_duck(con, file1, file2, "id_1, id_2")
    result["flat"] = {"polars": t_polars, "duck": t_duck}

    data1_struct, data2_struct = data_with_struct_id(data1, data2)
    if not from_file:
        t_polars = %timeit -o data1_struct.join(data2_struct, on = ["id"])
        t_duck = time_duck(con, data1_struct, data2_struct, "id")
    else:
        data1_struct.collect().write_parquet("data1_struct.parquet")
        data2_struct.collect().write_parquet("data2_struct.parquet")
        data1 = pl.scan_parquet("data1_struct.parquet")
        data2 = pl.scan_parquet("data2_struct.parquet")
        t_polars = %timeit -o data1.join(data2, on = ["id"]).collect()
        t_duck = time_duck(con, "data1_struct.parquet", "data2_struct.parquet", "id")

    result["struct"] = {"polars": t_polars, "duck": t_duck}
    return result

In [8]:
result = compare_timing(data1, data2, con)

467 μs ± 27.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.68 ms ± 200 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
472 μs ± 16.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.49 ms ± 79.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
result

{'flat': {'polars': <TimeitResult : 467 μs ± 27.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)>,
  'duck': <TimeitResult : 1.68 ms ± 200 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)>},
 'struct': {'polars': <TimeitResult : 472 μs ± 16.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)>,
  'duck': <TimeitResult : 1.49 ms ± 79.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)>}}

## Larger in-memory data

In [10]:
class FakeData(TestData):
    def make_data(self):
        self.sample_size = 1_000_000
        data1_dict = {
            "id_1": self.make_identifiers(1_000_000_000).astype(int),
            "id_2": self.make_identifiers(1_000_000_000).astype(int),
            "date": self.random_dates().astype(str),
            "wage": self.rng.random(size=self.sample_size).astype(np.float64),
            "sector": self.make_categorical(["1", "2", "3"], [0.2, 0.4, 0.4]),
            "null_col": self.make_null_col(),
        }
        data2_dict = {
            "id_1": data1_dict["id_1"],
            "id_2": data1_dict["id_2"],
            "date2": self.random_dates().astype(str),
            "wealth": self.rng.random(size=self.sample_size).astype(np.float64),
            "region": self.make_categorical(["1", "2", "3"], [0.2, 0.4, 0.4]),
        }

        return pl.DataFrame(data1_dict), pl.DataFrame(data2_dict)

    def make_parquet(self):
        data1, data2 = self.make_data()
        data1.write_parquet("data1.parquet")
        data2.write_parquet("data2.parquet")
        return "data1.parquet", "data2.parquet"

In [11]:
faker = FakeData()
data1, data2 = faker.make_data()

In [12]:
data1.head()

id_1,id_2,date,wage,sector,null_col
i64,i64,str,f64,str,f64
976699766,543643932,"""2025-09-23""",0.64845,"""1""",NaN
987940579,592235623,"""2014-09-10""",0.258209,"""2""",NaN
380195735,373289325,"""2001-12-13""",0.728351,"""3""",NaN
171359820,571704452,"""2021-03-07""",0.944631,"""3""",NaN
923246233,244441322,"""2005-05-28""",0.404038,"""1""",NaN


In [13]:
data2.head()

id_1,id_2,date2,wealth,region
i64,i64,str,f64,str
976699766,543643932,"""2014-10-04""",0.536864,"""1"""
987940579,592235623,"""2000-10-18""",0.737283,"""2"""
380195735,373289325,"""2018-01-11""",0.957396,"""3"""
171359820,571704452,"""2022-04-26""",0.274953,"""2"""
923246233,244441322,"""2008-10-07""",0.382603,"""1"""


In [14]:
result_large = compare_timing(data1, data2, con)

106 ms ± 13.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
107 ms ± 1.95 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
89.6 ms ± 2.02 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
107 ms ± 2.27 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Larger on-disk data

In [15]:
file1, file2 = faker.make_parquet()

In [16]:
type(file1)

str

In [17]:
result_from_file = compare_timing(file1, file2, con)

126 ms ± 5.08 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
143 ms ± 2.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
177 ms ± 3.32 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
151 ms ± 7.68 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
result_from_file

{'flat': {'polars': <TimeitResult : 126 ms ± 5.08 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
  'duck': <TimeitResult : 143 ms ± 2.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>},
 'struct': {'polars': <TimeitResult : 177 ms ± 3.32 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
  'duck': <TimeitResult : 151 ms ± 7.68 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>}}

## Joins between nested primary and nested secondary structs

#### In-memory

In [19]:
result_in_memory = {"flat": {}, "nested": {}}
result_from_file = {"flat": {}, "nested": {}}

In [20]:
class FakeDataNestedStructs(TestData):
    def make_data(self):
        self.sample_size = 1_000_000
        data1_dict = {
            "person_id": self.make_identifiers(1_000_000_000).astype(int),
            "school_id1": self.make_identifiers(1_000).astype(int),
            "school_id2": self.make_identifiers(2_000).astype(int),
            "company": self.make_identifiers(100_000).astype(int),
            "date": self.random_dates(),
            "wage": self.rng.random(size=self.sample_size).astype(np.float64),
            "sector": self.make_categorical(["1", "2", "3"], [0.2, 0.4, 0.4]),
            "null_col": self.make_null_col(),
        }
        data2_dict = {
            # schools are uniquely identified by school_id1 + school_id2
            "school_id1": data1_dict["school_id1"],
            "school_id2": data1_dict["school_id2"],
            "date2": self.random_dates(),
            "wealth": self.rng.random(size=self.sample_size).astype(np.float64),
            "region": self.make_categorical(["1", "2", "3"], [0.2, 0.4, 0.4]),
        }
        return pl.DataFrame(data1_dict), pl.DataFrame(data2_dict)

    def make_parquet(self):
        data1, data2 = self.make_data()
        data1.write_parquet("data1.parquet")
        data2.write_parquet("data2.parquet")
        return "data1.parquet", "data2.parquet"

In [21]:
faker = FakeDataNestedStructs()
data1, data2 = faker.make_data()
# NOTE: these are special cases - all ids are in both tables, and they are sorted in the same way.
# Thus, make the test more general?

In [22]:
data1.head()

person_id,school_id1,school_id2,company,date,wage,sector,null_col
i64,i64,i64,i64,date,f64,str,f64
28538927,957,1234,7711,2001-07-11,0.377129,"""1""",NaN
75816903,463,1279,21487,2008-02-07,0.088,"""2""",NaN
432302372,446,1441,11125,2013-07-03,0.85914,"""3""",NaN
48240786,829,1797,63861,2007-01-01,0.997841,"""1""",NaN
554133484,963,1831,78820,2012-10-13,0.478017,"""3""",NaN


In [23]:
data2.head()

school_id1,school_id2,date2,wealth,region
i64,i64,date,f64,str
957,1234,2010-08-13,0.466368,"""1"""
463,1279,2000-11-30,0.498901,"""2"""
446,1441,2025-01-07,0.409031,"""3"""
829,1797,2016-06-17,0.579342,"""2"""
963,1831,2012-07-10,0.731057,"""3"""


In [24]:
data1_struct = data1.with_columns(
    pl.struct(
        pl.struct(pl.col("school_id1").alias("key0"), pl.col("school_id2").alias("key1")).alias("school_id"),
        pl.col("company").alias("company_id"),
    ).alias("relations")
).drop(pl.col(["school_id1", "school_id2", "company"]))
data2_struct = data2.with_columns(
    pl.struct(pl.col("school_id1").alias("key0"), pl.col("school_id2").alias("key1")).alias("school_id")
).drop(pl.col(["school_id1", "school_id2"]))

In [25]:
data1_struct.head()

person_id,date,wage,sector,null_col,relations
i64,date,f64,str,f64,struct[2]
28538927,2001-07-11,0.377129,"""1""",NaN,"{{957,1234},7711}"
75816903,2008-02-07,0.088,"""2""",NaN,"{{463,1279},21487}"
432302372,2013-07-03,0.85914,"""3""",NaN,"{{446,1441},11125}"
48240786,2007-01-01,0.997841,"""1""",NaN,"{{829,1797},63861}"
554133484,2012-10-13,0.478017,"""3""",NaN,"{{963,1831},78820}"


In [26]:
data2_struct.head()

date2,wealth,region,school_id
date,f64,str,struct[2]
2010-08-13,0.466368,"""1""","{957,1234}"
2000-11-30,0.498901,"""2""","{463,1279}"
2025-01-07,0.409031,"""3""","{446,1441}"
2016-06-17,0.579342,"""2""","{829,1797}"
2012-07-10,0.731057,"""3""","{963,1831}"


In [27]:
result_in_memory["flat"][
    "polars"
] = %timeit -o data1.join(data2, left_on=["school_id1", "school_id2"], right_on=["school_id1", "school_id2"])

141 ms ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
result_in_memory["nested"][
    "polars"
] = %timeit -o data1_struct.join(data2_struct, left_on=pl.col("relations").struct.field("school_id"), right_on=pl.col("school_id"))

161 ms ± 2.73 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [29]:
con.execute("DROP TABLE IF EXISTS t1")
con.execute("DROP TABLE IF EXISTS t2")
con.execute("CREATE TABLE t1 AS SELECT * FROM data1_struct")
con.execute("CREATE TABLE t2 AS SELECT * FROM data2_struct")
# rel = con.sql(f"SELECT * FROM t1 INNER JOIN t2 ON t1.id.school_id=t2.school_id.school_id")

In [30]:
result_in_memory["nested"][
    "duckdb"
] = %timeit -o con.execute(f"SELECT * FROM t1 INNER JOIN t2 ON t1.relations.school_id=t2.school_id")

121 ms ± 2.57 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [31]:
con.execute("DROP TABLE IF EXISTS t1")
con.execute("DROP TABLE IF EXISTS t2")
con.execute("CREATE TABLE t1 AS SELECT * FROM data1")
con.execute("CREATE TABLE t2 AS SELECT * FROM data2")
rel = con.sql("SELECT * FROM t1 INNER JOIN t2 ON t1.school_id1=t2.school_id1 and t1.school_id2=t1.school_id2")
rel.limit(3)

┌───────────┬────────────┬────────────┬─────────┬────────────┬────────────────────┬─────────┬──────────┬────────────┬────────────┬────────────┬─────────────────────┬─────────┐
│ person_id │ school_id1 │ school_id2 │ company │    date    │        wage        │ sector  │ null_col │ school_id1 │ school_id2 │   date2    │       wealth        │ region  │
│   int64   │   int64    │   int64    │  int64  │    date    │       double       │ varchar │  double  │   int64    │   int64    │    date    │       double        │ varchar │
├───────────┼────────────┼────────────┼─────────┼────────────┼────────────────────┼─────────┼──────────┼────────────┼────────────┼────────────┼─────────────────────┼─────────┤
│ 147407032 │        957 │       1347 │   31420 │ 2019-01-06 │ 0.8105754004960977 │ 1       │      nan │        957 │       1234 │ 2010-08-13 │ 0.46636831796720535 │ 1       │
│  91207568 │        463 │       1552 │   54160 │ 2009-03-12 │ 0.6613962749675203 │ 1       │      nan │        463 │   

In [32]:
result_in_memory["flat"][
    "duckdb"
] = %timeit -o con.execute(f"SELECT * FROM t1 INNER JOIN t2 ON t1.school_id1=t2.school_id1 and t1.school_id2=t2.school_id2")

116 ms ± 2.17 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [33]:
result_in_memory

{'flat': {'polars': <TimeitResult : 141 ms ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)>,
  'duckdb': <TimeitResult : 116 ms ± 2.17 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>},
 'nested': {'polars': <TimeitResult : 161 ms ± 2.73 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
  'duckdb': <TimeitResult : 121 ms ± 2.57 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>}}

#### From file

In [34]:
data1.write_parquet("data1.parquet")
data2.write_parquet("data2.parquet")

In [35]:
data1 = pl.scan_parquet("data1.parquet")
data2 = pl.scan_parquet("data2.parquet")

In [36]:
data1.join(data2, left_on=["school_id1", "school_id2"], right_on=["school_id1", "school_id2"]).collect()

person_id,school_id1,school_id2,company,date,wage,sector,null_col,date2,wealth,region
i64,i64,i64,i64,date,f64,str,f64,date,f64,str
894250676,895,1098,95862,2005-03-23,0.608807,"""3""",NaN,2016-03-12,0.39472,"""2"""
70313330,893,868,72597,2024-09-14,0.157175,"""3""",NaN,2011-08-12,0.813579,"""3"""
70313330,893,868,72597,2024-09-14,0.157175,"""3""",NaN,2019-05-15,0.581283,"""1"""
432616482,697,740,71833,2012-06-20,0.677772,"""1""",NaN,2017-06-21,0.71851,"""1"""
432616482,697,740,71833,2012-06-20,0.677772,"""1""",NaN,2009-04-26,0.927433,"""2"""
…,…,…,…,…,…,…,…,…,…,…
789511561,823,741,27439,2024-10-19,0.305108,"""2""",NaN,2015-04-25,0.126929,"""3"""
77474257,909,1114,44758,2003-09-18,0.839738,"""1""",NaN,2010-03-22,0.746703,"""3"""
77474257,909,1114,44758,2003-09-18,0.839738,"""1""",NaN,2016-11-03,0.65983,"""1"""


In [37]:
result_from_file["flat"][
    "polars"
] = %timeit -o data1.join(data2, left_on=["school_id1", "school_id2"], right_on=["school_id1", "school_id2"]).collect()

148 ms ± 8.43 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [38]:
file1, file2 = ("data1.parquet", "data2.parquet")
sql = f"""
    SELECT * FROM read_parquet('{file1}') AS t1
    INNER JOIN read_parquet('{file2}') AS t2
    ON t1.school_id1=t2.school_id1 AND t1.school_id2=t1.school_id2
"""
rel = con.sql(sql)
rel.limit(3)

┌───────────┬────────────┬────────────┬─────────┬────────────┬─────────────────────┬─────────┬──────────┬────────────┬────────────┬────────────┬─────────────────────┬─────────┐
│ person_id │ school_id1 │ school_id2 │ company │    date    │        wage         │ sector  │ null_col │ school_id1 │ school_id2 │   date2    │       wealth        │ region  │
│   int64   │   int64    │   int64    │  int64  │    date    │       double        │ varchar │  double  │   int64    │   int64    │    date    │       double        │ varchar │
├───────────┼────────────┼────────────┼─────────┼────────────┼─────────────────────┼─────────┼──────────┼────────────┼────────────┼────────────┼─────────────────────┼─────────┤
│ 939881651 │         64 │        905 │   73883 │ 2010-02-10 │  0.5537990917675661 │ 2       │      nan │         64 │       1444 │ 2014-05-05 │  0.7795779913417027 │ 1       │
│ 368870128 │        431 │        735 │   87244 │ 2017-08-01 │ 0.04630852266333618 │ 2       │      nan │        43

In [39]:
result_from_file["flat"]["duckdb"] = %timeit -o con.execute(sql)

141 ms ± 3.53 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [40]:
data1_struct = data1.with_columns(
    pl.struct(
        pl.struct(pl.col("school_id1").alias("key0"), pl.col("school_id2").alias("key1")).alias("school_id"),
        pl.col("company").alias("company_id"),
    ).alias("relations")
).drop(pl.col(["school_id1", "school_id2", "company"]))
data2_struct = data2.with_columns(
    pl.struct(pl.col("school_id1").alias("key0"), pl.col("school_id2").alias("key1")).alias("school_id")
).drop(pl.col(["school_id1", "school_id2"]))

In [41]:
data1_struct.collect().write_parquet("data1_struct.parquet")
data2_struct.collect().write_parquet("data2_struct.parquet")

In [42]:
data1_struct = pl.scan_parquet("data1_struct.parquet")
data2_struct = pl.scan_parquet("data2_struct.parquet")

In [43]:
result_from_file["nested"][
    "polars"
] = %timeit -o data1_struct.join(data2_struct, left_on=pl.col("relations").struct.field("school_id"), right_on=pl.col("school_id")).collect()

276 ms ± 37.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [44]:
data1_struct.head().execute()

person_id,date,wage,sector,null_col,relations
i64,date,f64,str,f64,struct[2]
28538927,2001-07-11,0.377129,"""1""",NaN,"{{957,1234},7711}"
75816903,2008-02-07,0.088,"""2""",NaN,"{{463,1279},21487}"
432302372,2013-07-03,0.85914,"""3""",NaN,"{{446,1441},11125}"
48240786,2007-01-01,0.997841,"""1""",NaN,"{{829,1797},63861}"
554133484,2012-10-13,0.478017,"""3""",NaN,"{{963,1831},78820}"


In [45]:
data2_struct.head().execute()

date2,wealth,region,school_id
date,f64,str,struct[2]
2010-08-13,0.466368,"""1""","{957,1234}"
2000-11-30,0.498901,"""2""","{463,1279}"
2025-01-07,0.409031,"""3""","{446,1441}"
2016-06-17,0.579342,"""2""","{829,1797}"
2012-07-10,0.731057,"""3""","{963,1831}"


In [46]:
file1, file2 = ("data1_struct.parquet", "data2_struct.parquet")
sql = f"""
    SELECT * FROM read_parquet('{file1}') AS t1
    INNER JOIN read_parquet('{file2}') AS t2
    ON t1.relations.school_id=t2.school_id
"""
rel = con.sql(sql)
rel.limit(3)  # NOTE: different columns?

┌───────────┬────────────┬─────────────────────┬─────────┬──────────┬───────────────────────────────────────────────────────────────────────┬────────────┬─────────────────────┬─────────┬──────────────────────────────────┐
│ person_id │    date    │        wage         │ sector  │ null_col │                               relations                               │   date2    │       wealth        │ region  │            school_id             │
│   int64   │    date    │       double        │ varchar │  double  │ struct(school_id struct(key0 bigint, key1 bigint), company_id bigint) │    date    │       double        │ varchar │ struct(key0 bigint, key1 bigint) │
├───────────┼────────────┼─────────────────────┼─────────┼──────────┼───────────────────────────────────────────────────────────────────────┼────────────┼─────────────────────┼─────────┼──────────────────────────────────┤
│  33147382 │ 2020-12-13 │  0.9998517478936257 │ 3       │      nan │ {'school_id': {'key0': 64, 'key1': 1444}, 

In [47]:
result_from_file["nested"]["duckdb"] = %timeit -o con.execute(sql)

170 ms ± 4.11 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [48]:
result_in_memory

{'flat': {'polars': <TimeitResult : 141 ms ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)>,
  'duckdb': <TimeitResult : 116 ms ± 2.17 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>},
 'nested': {'polars': <TimeitResult : 161 ms ± 2.73 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
  'duckdb': <TimeitResult : 121 ms ± 2.57 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>}}

In [49]:
result_from_file

{'flat': {'polars': <TimeitResult : 148 ms ± 8.43 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>,
  'duckdb': <TimeitResult : 141 ms ± 3.53 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>},
 'nested': {'polars': <TimeitResult : 276 ms ± 37.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)>,
  'duckdb': <TimeitResult : 170 ms ± 4.11 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)>}}

**Penalty of nesting, in-memory; duckdb and polars**

In [70]:
np.mean(result_in_memory["nested"]["duckdb"].timings) / np.mean(result_in_memory["flat"]["duckdb"].timings)

np.float64(1.0407415269374358)

In [71]:
np.mean(result_in_memory["nested"]["polars"].timings) / np.mean(result_in_memory["flat"]["polars"].timings)

np.float64(1.1440881205297648)

**Penalty of nesting, from file, duckdb and polars**

In [72]:
np.mean(result_from_file["nested"]["duckdb"].timings) / np.mean(result_from_file["flat"]["duckdb"].timings)

np.float64(1.2022610946171912)

In [73]:
np.mean(result_from_file["nested"]["polars"].timings) / np.mean(result_from_file["flat"]["polars"].timings)

np.float64(1.8689441529925737)